# 1일차 — Tabular-based Methods

2026-07-27 (월) · 강화학습의 뼈대가 되는 MDP·동적계획법·시간차 학습을 표 기반 방법으로 익힙니다.

> 위에서부터 순서대로 실행하세요. 뒤 교시가 앞 교시의 변수·클래스를 그대로 이어 씁니다.


## 1교시 · 강화학습 소개

`09:30 ~ 10:30` · `bandit_epsilon_greedy.py`

- 강화학습이 지도학습·비지도학습과 어떻게 다른지 설명할 수 있다
- 에이전트-환경 상호작용 루프(상태·행동·보상)를 이해한다
- 탐험(Exploration)과 활용(Exploitation)의 트레이드오프를 이해한다


In [ ]:
import numpy as np

# 10개의 슬롯머신(Multi-Armed Bandit)으로 보는 탐험 vs 활용
np.random.seed(0)
n_arms = 10
true_means = np.random.normal(0, 1, n_arms)   # 각 팔의 실제 평균 보상

def run_bandit(epsilon, steps=2000):
    Q = np.zeros(n_arms)        # 행동가치 추정치
    N = np.zeros(n_arms)        # 각 팔을 당긴 횟수
    rewards = []
    for t in range(steps):
        if np.random.rand() < epsilon:
            a = np.random.randint(n_arms)      # 탐험
        else:
            a = np.argmax(Q)                   # 활용
        r = np.random.normal(true_means[a], 1) # 보상 샘플
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]              # 증분 평균 업데이트
        rewards.append(r)
    return np.mean(rewards)

for eps in [0.0, 0.01, 0.1, 0.5]:
    print(f"epsilon={eps:4.2f}  평균 보상 = {run_bandit(eps):.3f}")

# epsilon=0(탐험 없음)은 나쁜 팔에 갇히고,
# 0.5(과도한 탐험)는 보상을 낭비합니다. 0.1 근처가 균형점.

## 2교시 · MDP 소개

`10:30 ~ 11:30` · `gridworld_mdp.py`

- 마르코프 결정 과정(MDP)의 5요소 (S, A, P, R, γ)를 설명할 수 있다
- 상태가치함수 V(s)와 행동가치함수 Q(s,a)의 차이를 이해한다
- 벨만 방정식의 재귀 구조를 이해한다


In [ ]:
import numpy as np

# 4x4 GridWorld MDP 정의 — 이후 세션에서 계속 재사용합니다
# 상태: 0~15 (왼쪽 위에서 오른쪽 아래로), 0과 15는 종료 상태
# 행동: 0=상, 1=하, 2=좌, 3=우 / 보상: 이동마다 -1

N = 4
n_states = N * N
n_actions = 4
TERMINALS = [0, n_states - 1]

def step(s, a):
    """결정적 전이: (다음상태, 보상) 반환"""
    if s in TERMINALS:
        return s, 0
    r, c = divmod(s, N)
    if a == 0: r = max(r - 1, 0)
    elif a == 1: r = min(r + 1, N - 1)
    elif a == 2: c = max(c - 1, 0)
    elif a == 3: c = min(c + 1, N - 1)
    return r * N + c, -1

# 전이 텐서 P[s][a] = (s', r) 를 미리 만들어 두면 DP가 간단해집니다
P = [[step(s, a) for a in range(n_actions)] for s in range(n_states)]

# 무작위 정책의 한 에피소드 시뮬레이션
s, trajectory = 5, []
rng = np.random.default_rng(42)
while s not in TERMINALS:
    a = rng.integers(n_actions)
    s_next, r = P[s][a]
    trajectory.append((s, a, r))
    s = s_next
print(f"에피소드 길이: {len(trajectory)}, 총 보상: {sum(t[2] for t in trajectory)}")

## 3교시 · Dynamic Programming 소개

`11:30 ~ 12:30` · `policy_evaluation.py`

- 정책 평가(Policy Evaluation)와 정책 개선(Policy Improvement)을 구분한다
- 정책 반복(PI)과 가치 반복(VI)의 차이를 설명할 수 있다
- DP가 model-based 방법인 이유와 한계를 이해한다


In [ ]:
import numpy as np

# 무작위 정책에 대한 반복적 정책 평가 (4x4 GridWorld, 앞 세션의 P 재사용)
gamma = 1.0
theta = 1e-6          # 수렴 판정 기준

V = np.zeros(n_states)
iteration = 0
while True:
    delta = 0.0
    V_new = V.copy()
    for s in range(n_states):
        if s in TERMINALS:
            continue
        # 무작위 정책: 4방향 각 0.25 확률
        v = 0.0
        for a in range(n_actions):
            s_next, r = P[s][a]
            v += 0.25 * (r + gamma * V[s_next])
        V_new[s] = v
        delta = max(delta, abs(v - V[s]))
    V = V_new
    iteration += 1
    if delta < theta:
        break

print(f"{iteration}회 반복 후 수렴")
print(np.round(V.reshape(4, 4), 1))
# 종료 상태에서 멀수록 가치가 낮아지는(-22 근처) 것을 확인하세요

## 4교시 · Policy Iteration, Value Iteration 구현

`13:30 ~ 14:30` · `pi_vi_gridworld.py`

- 4x4 GridWorld에서 정책 반복을 NumPy로 구현한다
- 가치 반복을 구현하고 두 방법의 수렴 결과를 비교한다


In [ ]:
import numpy as np

gamma = 1.0

def q_from_v(V, s):
    """상태 s에서 각 행동의 Q값 계산"""
    return np.array([P[s][a][1] + gamma * V[P[s][a][0]]
                     for a in range(n_actions)])

# ── 정책 반복 (Policy Iteration) ──────────────────
def policy_iteration():
    policy = np.zeros(n_states, dtype=int)      # 모든 상태에서 행동 0
    while True:
        # 1) 정책 평가
        # 평가 sweep에 상한을 둡니다 (MAX_SWEEP).
        # 이유: 초기 정책(모두 '상')은 맨 윗줄에서 벽에 막혀 제자리에 머뭅니다.
        # 그러면 V[s] = -1 + 1.0 * V[s] 라서 gamma=1일 때 값이 발산해
        # delta < 1e-6 조건이 영원히 성립하지 않습니다 (무한 루프).
        # 상한을 두고 개선 단계로 넘어가면 다음 정책은 종료 상태에 도달하므로
        # 이후로는 정상 수렴합니다 — 이를 modified policy iteration이라 부릅니다.
        MAX_SWEEP = 1000
        V = np.zeros(n_states)
        for _ in range(MAX_SWEEP):
            delta = 0.0
            for s in range(n_states):
                if s in TERMINALS: continue
                s_next, r = P[s][policy[s]]
                v = r + gamma * V[s_next]
                delta = max(delta, abs(v - V[s]))
                V[s] = v
            if delta < 1e-6: break
        # 2) 정책 개선
        stable = True
        for s in range(n_states):
            if s in TERMINALS: continue
            best_a = np.argmax(q_from_v(V, s))
            if best_a != policy[s]:
                stable = False
                policy[s] = best_a
        if stable:
            return policy, V

# ── 가치 반복 (Value Iteration) ───────────────────
def value_iteration():
    V = np.zeros(n_states)
    while True:
        delta = 0.0
        for s in range(n_states):
            if s in TERMINALS: continue
            v = q_from_v(V, s).max()            # max가 곧 개선
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < 1e-6: break
    policy = np.array([np.argmax(q_from_v(V, s)) for s in range(n_states)])
    return policy, V

arrows = np.array(['↑', '↓', '←', '→'])
pi_policy, pi_V = policy_iteration()
vi_policy, vi_V = value_iteration()
print("PI 최적 정책:"); print(arrows[pi_policy].reshape(4, 4))
print("VI 최적 정책:"); print(arrows[vi_policy].reshape(4, 4))
print("두 가치함수 일치:", np.allclose(pi_V, vi_V))

## 5교시 · Monte-Carlo 방법, Temporal Difference 방법 소개

`14:30 ~ 15:30` · `mc_vs_td_prediction.py`

- 모델 없이(model-free) 가치를 추정하는 두 접근을 이해한다
- MC의 낮은 편향·높은 분산, TD의 부트스트래핑·낮은 분산 특성을 비교한다
- TD(0) 업데이트 식을 쓸 수 있다


In [ ]:
import numpy as np                      # 숫자 계산을 도와주는 도구

# 같은 문제를 두 가지 방법으로 풀어 보고 결과를 비교합니다.
#   방법 1) 몬테카를로 — 한 판을 끝까지 하고 나서 배우기
#   방법 2) 시간차(TD)  — 한 걸음 옮길 때마다 바로 배우기

rng = np.random.default_rng(0)         # 무작위 뽑기 도구. 0은 '항상 같은 결과'를 위한 값
alpha = 0.05                           # 학습률 — 새로 안 것을 얼마나 믿을지 (0~1)
gamma = 1.0                            # 미래를 얼마나 챙길지 (1이면 먼 미래도 그대로)


def gen_episode():
    """한 판을 끝까지 해보고, 지나온 기록을 돌려줍니다."""
    s = rng.integers(1, n_states - 1)  # 출발 칸을 아무 데나 고름 (끝 칸 제외)
    episode = []                       # 지나온 기록을 담을 빈 목록

    while s not in TERMINALS:          # 끝 칸에 도착할 때까지 반복
        a = rng.integers(n_actions)    # 아무 방향이나 하나 고름 (무작위로 걷기)
        s_next, r = P[s][a]            # 그 방향으로 가면 어디로 가고 몇 점인지
        episode.append((s, r))         # "어느 칸에서 몇 점 받았다"를 기록
        s = s_next                     # 다음 칸으로 이동

    return episode                     # 한 판의 기록 전체를 돌려줌


# ── 방법 1) 몬테카를로 — 끝까지 하고 나서 배우기 ──────────
V_mc = np.zeros(n_states)              # 각 칸의 값. 처음엔 전부 0 (아무것도 모름)

for _ in range(5000):                  # 5000판 반복
    episode = gen_episode()            # 한 판을 끝까지 해봄
    G = 0.0                            # 이 지점부터 끝까지 받은 총 점수
    visited = set()                    # 이번 판에서 이미 들른 칸 목록

    # 뒤에서부터 되짚습니다. 끝에서부터 세야 "여기부터 끝까지"가 계산됩니다.
    for s, r in reversed(episode):
        G = r + gamma * G              # (지금 점수) + (여기 다음부터 끝까지)

        if s not in visited:           # 같은 칸을 여러 번 지났으면 첫 번째만 사용
            visited.add(s)             # 들렀다고 표시
            # 지금 알던 값(V_mc[s])을 실제 결과(G) 쪽으로 조금(alpha) 옮깁니다.
            V_mc[s] += alpha * (G - V_mc[s])


# ── 방법 2) 시간차(TD) — 한 걸음마다 바로 배우기 ──────────
V_td = np.zeros(n_states)              # 마찬가지로 전부 0에서 시작

for _ in range(5000):                  # 5000판 반복
    s = rng.integers(1, n_states - 1)  # 아무 칸에서 출발

    while s not in TERMINALS:          # 끝 칸에 닿을 때까지
        a = rng.integers(n_actions)    # 아무 방향이나 고름
        s_next, r = P[s][a]            # 가보니 어디로 갔고 몇 점인지

        # 여기가 몬테카를로와 다른 곳입니다.
        # 끝까지 안 가고, "지금 점수 + 다음 칸의 (아직 부정확한) 값"으로 대신합니다.
        td_error = r + gamma * V_td[s_next] - V_td[s]

        V_td[s] += alpha * td_error    # 그 차이만큼 조금 옮김
        s = s_next                     # 다음 칸으로
        s = s_next

print("MC 추정:"); print(np.round(V_mc.reshape(4, 4), 1))
print("TD 추정:"); print(np.round(V_td.reshape(4, 4), 1))
# 둘 다 DP 정답(-14, -20, -22...)에 근접하는지 확인하세요

## 6교시 · SARSA와 Q-Learning 소개

`15:30 ~ 16:30` · `update_rules.py`

- TD 제어에서 SARSA와 Q-Learning의 업데이트 식을 구분한다
- On-policy와 Off-policy의 차이를 설명할 수 있다


In [ ]:
# 두 알고리즘의 차이는 단 한 줄 — TD 목표(target)의 정의

# Q 는 '표'입니다. Q[칸][방향] = 그 칸에서 그 방향으로 가면 얼마나 좋은지.
# 아래 두 함수는 그 표의 숫자 하나를 고치는 방법입니다.
# 두 함수는 딱 한 줄만 다릅니다. 그 줄을 잘 보세요.


# ── 방법 1) SARSA — 내가 "실제로 할" 행동을 보고 고침 ──────
def sarsa_update(Q, s, a, r, s_next, a_next, alpha=0.1, gamma=0.99):
    # s      : 지금 있는 칸
    # a      : 지금 하려는 행동
    # r      : 그 행동으로 받은 점수
    # s_next : 그래서 가게 된 다음 칸
    # a_next : 다음 칸에서 "실제로 할" 행동   ← SARSA 는 이게 필요합니다
    # alpha  : 얼마나 믿을지 (0.1이면 10%만 반영)
    # gamma  : 미래를 얼마나 챙길지

    # 목표값 = 지금 받은 점수 + 다음 칸에서 실제로 할 행동의 값
    target = r + gamma * Q[s_next][a_next]

    # 지금 알던 값을 목표값 쪽으로 alpha 만큼 조금 옮깁니다.
    # (목표값 - 지금값) 이 '내 예상이 얼마나 빗나갔나' 입니다.
    Q[s][a] += alpha * (target - Q[s][a])


# ── 방법 2) Q-러닝 — "제일 좋은" 행동을 보고 고침 ─────────
def q_learning_update(Q, s, a, r, s_next, alpha=0.1, gamma=0.99):
    # 여기는 a_next 가 없습니다. 다음에 뭘 할지 몰라도 되기 때문입니다.

    # 목표값 = 지금 받은 점수 + 다음 칸에서 "가장 좋은" 행동의 값
    # max 는 여러 값 중 제일 큰 것을 고르는 것입니다.
    # 실제로 그 행동을 할지는 상관하지 않습니다. "만약 최선을 다한다면" 을 가정합니다.
    target = r + gamma * max(Q[s_next])

    Q[s][a] += alpha * (target - Q[s][a])    # 고치는 방식은 위와 똑같습니다


# ── 행동 고르기 — 두 방법이 공통으로 씁니다 ───────────────
import random                                # 무작위 뽑기 도구


def epsilon_greedy(Q, s, n_actions, epsilon=0.1):
    # epsilon 은 '아무거나 해볼 확률' 입니다. 0.1이면 10번에 1번.
    # 왜 일부러 아무거나 할까요?
    #   처음 우연히 괜찮았던 길만 계속 가면 더 좋은 길을 영영 못 찾기 때문입니다.

    if random.random() < epsilon:            # 0~1 사이 아무 숫자를 뽑아서
        return random.randrange(n_actions)   # epsilon 보다 작으면 → 아무 방향이나

    # 그렇지 않으면 → 지금까지 알기로 가장 좋은 방향
    # key=... 는 "이 기준으로 가장 큰 것을 고르라" 는 뜻입니다.
    return max(range(n_actions), key=lambda a: Q[s][a])

## 7교시 · SARSA와 Q-Learning 구현

`16:30 ~ 17:30` · `cliff_sarsa_qlearning.py`

- Gymnasium CliffWalking 환경에서 SARSA와 Q-Learning을 완성한다
- 두 알고리즘이 학습한 경로의 차이를 직접 확인한다


In [ ]:
import gymnasium as gym
import numpy as np

# CliffWalking = '절벽 걷기' 놀이판입니다.
# 4줄 x 12칸 격자이고, 아래쪽 가운데가 절벽입니다.
# 한 걸음마다 -1점, 절벽에 빠지면 -100점을 받고 출발점으로 돌아갑니다.
# 목표는 오른쪽 아래 끝에 도착하는 것입니다.
env = gym.make("CliffWalking-v1")

# 이 놀이판에 칸이 몇 개인지, 방향이 몇 개인지 물어봅니다.
n_states = env.observation_space.n      # 칸의 개수 (48개)
n_actions = env.action_space.n          # 방향의 개수 (4개: 상하좌우)

alpha = 0.1        # 학습률 — 새로 안 것을 10%만 반영
gamma = 0.99       # 미래를 얼마나 챙길지
epsilon = 0.1      # 아무거나 해볼 확률 — 10번에 1번

rng = np.random.default_rng(0)          # 무작위 뽑기 도구 (0은 매번 같은 결과용)


def eps_greedy(Q, s):
    """어느 방향으로 갈지 고릅니다."""
    if rng.random() < epsilon:          # 10% 확률로
        return rng.integers(n_actions)  #   → 아무 방향이나 (새로운 길 찾기)
    return int(np.argmax(Q[s]))         # 90% 는 지금까지 가장 좋았던 방향


def train(method, episodes=500):
    """500판을 하면서 표(Q)를 채워 나갑니다.

    method 가 "sarsa" 면 SARSA, 아니면 Q-러닝으로 학습합니다.
    두 방법의 차이는 아래 if 문 한 곳뿐입니다.
    """
    # 표를 0으로 채워 시작합니다. 세로 48칸 x 가로 4방향.
    Q = np.zeros((n_states, n_actions))
    returns = []                        # 판마다 받은 총점을 기록할 목록

    for _ in range(episodes):           # 500판 반복
        s, _ = env.reset()              # 출발점으로 돌아가 새 판 시작
        a = eps_greedy(Q, s)            # 첫 방향을 미리 정해 둡니다
        total = 0                       # 이번 판에서 받은 총점
        done = False                    # 판이 끝났는지

        while not done:                 # 끝날 때까지 반복
            # 정한 방향으로 한 걸음 갑니다.
            #   s_next : 가게 된 칸
            #   r      : 받은 점수 (보통 -1, 절벽이면 -100)
            #   term   : 목표에 도착했거나 절벽에 빠졌는지
            #   trunc  : 너무 오래 걸려 강제로 끊겼는지
            s_next, r, term, trunc, _ = env.step(a)
            done = term or trunc

            # ── 두 방법이 갈리는 딱 한 곳 ──────────────────
            if method == "sarsa":
                # SARSA: 다음 칸에서 "실제로 할" 방향을 먼저 정하고,
                #        그 방향의 값을 목표에 씁니다.
                #        가끔 딴 길로 새는 것까지 계산에 들어갑니다.
                a_next = eps_greedy(Q, s_next)
                target = r + gamma * Q[s_next][a_next] * (not done)
            else:
                # Q-러닝: 다음 칸에서 "가장 좋은" 방향의 값을 씁니다.
                #        실제로 그 방향으로 갈지는 상관없습니다.
                target = r + gamma * Q[s_next].max() * (not done)
                a_next = eps_greedy(Q, s_next)

            # (not done) 이 붙은 이유:
            #   판이 끝났으면 앞으로 받을 점수가 없으므로 뒤쪽을 0으로 만듭니다.
            #   이걸 빼먹으면 끝난 뒤에도 점수가 계속 더해져 숫자가 이상해집니다.

            # 표의 숫자 하나를 고칩니다. (목표 - 지금값) 만큼 조금 옮기기.
            Q[s][a] += alpha * (target - Q[s][a])

            # 다음 걸음을 위해 자리를 옮깁니다.
            s = s_next
            a = a_next
            total = total + r

        returns.append(total)           # 이번 판 총점을 기록

    return Q, returns

Q_sarsa, ret_s = train("sarsa")
Q_qlearn, ret_q = train("qlearning")
print(f"SARSA      마지막 100ep 평균 보상: {np.mean(ret_s[-100:]):.1f}")
print(f"Q-Learning 마지막 100ep 평균 보상: {np.mean(ret_q[-100:]):.1f}")

# greedy 경로 시각화: SARSA는 위쪽 안전 경로, Q-Learning은 절벽 옆 최단 경로
for name, Q in [("SARSA", Q_sarsa), ("Q-Learning", Q_qlearn)]:
    grid = np.full(48, '.', dtype=str)
    s, _ = env.reset()
    for _ in range(30):
        a = int(np.argmax(Q[s]))
        grid[s] = '*'
        s, r, term, trunc, _ = env.step(a)
        if term or trunc: break
    print(f"\n[{name} greedy 경로]")
    print('\n'.join(''.join(row) for row in grid.reshape(4, 12)))